# Parallel Execution Deep Dive

This notebook demonstrates agentflowkit's parallel DAG execution:
- How `_resolve_levels()` partitions agents into parallel groups
- Measurable performance difference between parallel and sequential execution
- Timeout enforcement demo
- Cache hit rate demonstration

All examples use a **mock LLM** so no API key is required.

In [ ]:
import asyncio
import time
from agentflow import Agent, AgentResult, Pipeline
from agentflow.agent import BaseAgent
from agentflow.exceptions import AgentTimeoutError

## Mock LLM & Slow Agent (no API key needed)

In [ ]:
class MockLLM:
    """Simulates an LLM that takes a fixed time to respond."""
    def __init__(self, latency: float = 0.3):
        self.latency = latency
    
    async def generate(self, messages, **kwargs):
        await asyncio.sleep(self.latency)  # simulate API latency
        return {"content": f"response", "tokens": 50, "duration": self.latency, "model": "mock", "cached": False}


class TimedAgent(BaseAgent):
    """Agent that records when it started and finished."""
    def __init__(self, name: str, sleep_s: float = 0.3):
        super().__init__(name=name, role=f"{name} role")
        self.sleep_s = sleep_s
        self.start_at = None
        self.finish_at = None
    
    async def execute(self, task, context, llm):
        self.start_at = time.perf_counter()
        await asyncio.sleep(self.sleep_s)
        self.finish_at = time.perf_counter()
        return AgentResult(agent=self.name, output=f"output-{self.name}", tokens_used=50, duration=self.sleep_s)

print("Mock infrastructure ready.")

## 1. DAG Level Resolution

Let's inspect how `_resolve_levels()` partitions a diamond DAG.

In [ ]:
# Diamond: A → [B, C] → D
pipe = Pipeline(llm=MockLLM())
a = TimedAgent("A")
b = TimedAgent("B")
c = TimedAgent("C")
d = TimedAgent("D")

pipe.add(a)
pipe.add(b, depends_on=["A"])
pipe.add(c, depends_on=["A"])
pipe.add(d, depends_on=["B", "C"])

levels = pipe._resolve_levels()
for i, level in enumerate(levels):
    agents_in_level = [n.agent.name for n in level]
    can_parallel = len(agents_in_level) > 1
    print(f"Level {i}: {agents_in_level}  {'← PARALLEL' if can_parallel else ''}")

## 2. Sequential vs Parallel Benchmark

Two agents each taking 0.3s:
- **Sequential** (with dependency): total ≈ 0.6s
- **Parallel** (no dependency): total ≈ 0.3s

In [ ]:
llm = MockLLM(latency=0.0)  # zero LLM latency, agent sleep dominates

# --- Sequential (A → B) ---
s1 = TimedAgent("seq_a", sleep_s=0.3)
s2 = TimedAgent("seq_b", sleep_s=0.3)

seq_pipe = Pipeline(llm=llm)
seq_pipe.add(s1)
seq_pipe.add(s2, depends_on=["seq_a"])  # creates a dependency

t0 = time.perf_counter()
await seq_pipe.run("task")
seq_time = time.perf_counter() - t0

# --- Parallel (A ∥ B) ---
p1 = TimedAgent("par_a", sleep_s=0.3)
p2 = TimedAgent("par_b", sleep_s=0.3)

par_pipe = Pipeline(llm=llm)
par_pipe.add(p1)  # no dependency
par_pipe.add(p2)  # no dependency → runs in parallel

t0 = time.perf_counter()
await par_pipe.run("task")
par_time = time.perf_counter() - t0

speedup = seq_time / par_time
print(f"Sequential: {seq_time:.3f}s")
print(f"Parallel:   {par_time:.3f}s")
print(f"Speedup:    {speedup:.1f}x")
assert par_time < seq_time * 0.7, f"Parallel should be significantly faster!"

## 3. Visualize Concurrent Execution

In [ ]:
# Build a 5-agent parallel pipeline and visualize their execution windows
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

agents = [TimedAgent(f"agent_{i}", sleep_s=0.2 + i*0.05) for i in range(5)]

viz_pipe = Pipeline(llm=llm)
for agent in agents:
    viz_pipe.add(agent)  # all independent → all parallel

reference = time.perf_counter()
await viz_pipe.run("task")

fig, ax = plt.subplots(figsize=(10, 3))
colors = plt.cm.tab10.colors

for i, agent in enumerate(agents):
    start = agent.start_at - reference
    end = agent.finish_at - reference
    ax.barh(i, end - start, left=start, height=0.6, color=colors[i], alpha=0.8)
    ax.text(start + (end - start) / 2, i, agent.name, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

ax.set_xlabel("Wall time (seconds)")
ax.set_title("5 parallel agents — concurrent execution")
ax.set_yticks([])
ax.set_xlim(-0.02)
plt.tight_layout()
plt.show()

total_wall = max(a.finish_at for a in agents) - min(a.start_at for a in agents)
total_sum = sum(a.sleep_s for a in agents)
print(f"Wall time: {total_wall:.3f}s")
print(f"Sum of all agent times: {total_sum:.3f}s")
print(f"Efficiency: {total_sum / total_wall:.1f}x")

## 4. Timeout Enforcement

In [ ]:
slow = TimedAgent("slow_agent", sleep_s=2.0)

timeout_pipe = Pipeline(llm=llm)
timeout_pipe.add(slow, timeout=0.5)  # give it only 0.5s

try:
    await timeout_pipe.run("task")
    print("ERROR: should have timed out!")
except AgentTimeoutError as e:
    print(f"Caught AgentTimeoutError: agent='{e.agent_name}', timeout={e.timeout_seconds}s")

## 5. Cache Hit Rate Demonstration

In [ ]:
import os
from agentflow import InMemoryCache, LLM
from unittest.mock import AsyncMock, MagicMock

# Create LLM with cache
cache = InMemoryCache()
cached_llm = LLM(model="gpt-4o-mini", api_key="fake", cache=cache)

# Mock the underlying API client
call_count = {"n": 0}

def make_mock_response():
    call_count["n"] += 1
    mock = MagicMock()
    mock.choices = [MagicMock()]
    mock.choices[0].message.content = f"response #{call_count['n']}"
    mock.usage.total_tokens = 10
    mock.model = "gpt-4o-mini"
    return mock

cached_llm._client = MagicMock()
cached_llm._client.chat.completions.create = AsyncMock(side_effect=lambda **kwargs: make_mock_response())

messages = [{"role": "user", "content": "What is 2+2?"}]

results = []
for i in range(5):
    r = await cached_llm.generate(messages)
    results.append(r)

print(f"API calls made: {call_count['n']} (expected: 1)")
print(f"Cache hits: {sum(1 for r in results if r['cached'])} / {len(results)}")
for i, r in enumerate(results):
    print(f"  Call {i+1}: cached={r['cached']}, content='{r['content']}'")